In [0]:
%python
bronze_path = '/Volumes/workspace/techvenda/filestore/bronze/'
silver_path = '/Volumes/workspace/techvenda/filestore/silver/'
gold_path = '/Volumes/workspace/techvenda/filestore/gold/'
origem_path = '/Volumes/workspace/techvenda/filestore/origem/'

In [0]:
%python
#Tabelas temporarias
silver_mapeamento= {
    'temp_silver_clientes' : f'{silver_path}/clientes/',
    'temp_silver_itens_pedido' : f'{silver_path}/itens_pedido/',
    'temp_silver_pedidos' : f'{silver_path}/pedidos/',
    'temp_silver_produtos' : f'{silver_path}/produtos/',
    'temp_silver_vendedores' : f'{silver_path}/vendedores/',
    'temp_silver_pedidos_enriquecidos' : f'{silver_path}/pedidos_enriquecidos/'

}
for view_name, path in silver_mapeamento.items():
    (spark.read.format('delta')
        .load(path)
        .createOrReplaceTempView(view_name)
    )

In [0]:
%sql
select * from temp_silver_produtos

In [0]:
%sql
select * from temp_silver_pedidos_enriquecidos

In [0]:
%python
top_produtos_categoria = spark.sql("""
    with vendas_categoria_produto as (
        select 
            categoria,
            nome_produto,
            SUM(quantidade) as quantidade_vendida,
            ROUND(SUM(preco_unitario * quantidade), 2) as receita_produto
            
    from temp_silver_pedidos_enriquecidos
    group by categoria, nome_produto
    ),
    ranking_produtos as (
        select 
            categoria,
            nome_produto,
            quantidade_vendida,
            receita_produto,
            RANK() OVER(PARTITION BY categoria 
            order by receita_produto desc) as ranking
        from vendas_categoria_produto
    )
    select 
        categoria,
        nome_produto,
        quantidade_vendida,
        receita_produto,
        ranking  
    
    from ranking_produtos
    where ranking <= 3
    order by categoria, ranking 
                            """)

# salvar em Delta na Gold
top_produtos_categoria.write\
    .mode('overwrite')\
        .format('delta')\
            .option('mergeShema','true')\
                .save(f'{gold_path}/top_produtos_categoria/')

In [0]:
%sql
create table if not exists workspace.techvenda.top_produtos_categoria 
select * from delta. `/Volumes/workspace/techvenda/filestore/gold/top_produtos_categoria/`

In [0]:
%sql
select * from workspace.techvenda.top_produtos_categoria